## Suggested Tools and Resources for Scoring Oracles

### MolMIM NIM Endpoint
**Link**: [NVIDIA MolMIM NIM Documentation](https://docs.nvidia.com/nim/)

**Installation**
Installation and usage of the MolMIM NIM is outlined in the tutorial notebooks in the [`../tutorials`](../tutorials) directory.

### RDKit - Cheminformatics Toolkit
**Link**: [RDKit Official Website](https://www.rdkit.org/)
**Installation**:
```bash
# Via pip (recommended)
pip install rdkit

# Via conda
conda install -c conda-forge rdkit
```
**Common Scoring Tools & Usage Examples**:
```python
from rdkit import Chem
from rdkit.Chem import QED, Descriptors, Crippen

# Load molecule
mol = Chem.MolFromSmiles("CCO")

# Drug-likeness (QED)
qed_score = QED.qed(mol)

# Synthetic Accessibility Score
# see https://greglandrum.github.io/rdkit-blog/posts/2023-12-01-using_sascore_and_npscore.html
# for additional background
import sys
import os
sys.path.append(os.path.join(os.environ['CONDA_PREFIX'],'share','RDKit','Contrib'))

from SA_Score import sascorer
sascorer.calculateScore(mol)

# Lipophilicity (LogP)
logp = Crippen.MolLogP(mol)

# Molecular Weight
mw = Descriptors.MolWt(mol)
```

### UniDock - Molecular Docking
**Link**: [UniDock GitHub Repository](https://github.com/dptech-corp/Uni-Dock)
**Installation**:
```bash
# Clone repository
git clone https://github.com/dptech-corp/Uni-Dock.git
cd Uni-Dock/unidock_tools
pip install .
```

See step-by-step instructions for running a UniDock pipeline at [https://github.com/dptech-corp/Uni-Dock/tree/main/unidock_tools#applications](https://github.com/dptech-corp/Uni-Dock/tree/main/unidock_tools#applications)


### Integration Example for Scoring Oracle using RDKit
```python
def comprehensive_scoring_oracle(smiles):
    """
    Combine multiple scoring tools for comprehensive evaluation
    """
    from rdkit import Chem
    from rdkit.Chem import QED, Descriptors
    
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return 0.0
    
    # Drug-likeness
    qed_score = QED.qed(mol)
    
    # Molecular properties
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    
    # Combine scores (customize weights as needed)
    combined_score = (qed_score * 0.5 + 
                     (1.0 if 150 <= mw <= 500 else 0.0) * 0.3 +
                     (1.0 if -0.4 <= logp <= 5.6 else 0.0) * 0.2)
    
    return combined_score
```

### Example Scoring Workflow (CDK Inhibitor Challenge)

The repository includes a ready-to-run scoring pipeline in `challenge/example-scoring/`:

- `evaluate_submission.py`: end-to-end evaluator that combines Boltz2 affinity predictions, selectivity metrics, CDK11 avoidance, PAINS filtering, novelty scoring, and composite score ranking.
- `README.md`: usage guide with prerequisites, command examples, and flag descriptions.

Run the workflow from the repo root:

```bash
python challenge/example-scoring/evaluate_submission.py scoring/demo_compounds.csv DemoTeam \
  --output-dir challenge/example-scoring/results
```

#### Chemical Novelty Scoring
- The evaluator looks for a ChEMBL fingerprint cache at `scoring/chembl_data/chembl_fingerprints.pkl`.
- Override the location with `--chembl-path /path/to/cache_dir` (directory must contain `chembl_fingerprints.pkl`).
- If the cache is absent, every compound is treated as novel (novelty score = 1.0). See `scoring/README_evaluation_script.md` for cache generation steps.

### Deploying the Boltz2 NIM Endpoint Locally

Boltz2 produces the pIC50/IC50 values that drive the potency and selectivity metrics.

1. **Authenticate with NVIDIA NGC**
   ```bash
   ngc config set
   ```
   Enter your NGC API key when prompted.
2. **Fetch the container image**
   ```bash
   docker pull nvcr.io/nim/mit/boltz2:latest
   ```
3. **Launch the service on localhost (default evaluator port 8000)**
   ```bash
   docker run --rm -it \
     -p 8000:8000 \
     -e NIM_API_KEY="<your_api_key>" \
     nvcr.io/nim/mit/boltz2:latest
   ```
   - Adjust the host port if 8000 is unavailable.
   - Point the evaluator at another URL with `--boltz2-url`.
4. **Verify the endpoint**
   ```bash
   curl http://localhost:8000/v1/health
   ```

With the container running you can evaluate compounds with live affinity predictions:

```bash
python challenge/example-scoring/evaluate_submission.py my_compounds.csv MyTeam \
  --boltz2-url http://localhost:8000 \
  --confidence-threshold 0.2
```



# Licensing

Copyright © 2025 OpenACC-Standard.org.  This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.